# setup + imports

In [13]:
import pandas as pd
import numpy as np
logs = pd.read_csv('sector_logs.csv')
queries = pd.read_csv('query_states.csv')

In [20]:
N, A, GAMMA = 1000, 4, 0.99
SHIFT = [1, -1, 50, -50]                  # what each action does (found in the log)
GAMBLE_VALUE = 0.2 * 300 + 0.8 * (-100)

# data exploration

In [21]:
counts = {}
for s, a, r, s2 in logs.values:
    key = (int(a), int(s2) - int(s))      # (action, change in square number)
    counts[key] = counts.get(key, 0) + 1

for key in sorted(counts):
    if counts[key] > 20:                  # skip the rare jump differences
        print(key, counts[key])

(0, 1) 891
(1, -1) 914
(2, -950) 44
(2, 50) 888
(3, -50) 921
(3, 950) 55


In [22]:
counts = {}
for r in logs.reward:
    key = round(r, -1)                    # round to nearest 10
    counts[key] = counts.get(key, 0) + 1
print(counts)

{-0.0: 3700, -100.0: 1830, 300.0: 459, -50.0: 10, 50.0: 1}


In [23]:
'''
for s, a, r, s2 in logs.values:
    if r > 100 or r < -70:
        print(int(s), int(a), r, int(s2))
'''

'\nfor s, a, r, s2 in logs.values:\n    if r > 100 or r < -70:\n        print(int(s), int(a), r, int(s2))\n'

In [27]:
kind = -np.ones((N, A), dtype=int)
for s, a, r, s2 in logs.values:
    s, a, s2 = int(s), int(a), int(s2)
    is_gamble = (s2 == 0 and r < -70) or (s2 == 999 and r > 100)
    kind[s, a] = 1 if is_gamble else 0

In [28]:
nxt = np.zeros((N, A), dtype=int)
rew = np.zeros((N, A))
for s in range(N):
    for a in range(A):
        s2 = (s + SHIFT[a]) % N
        nxt[s, a] = s2
        if s2 == 999:
            rew[s, a] = 50
        elif s2 <= 9:
            rew[s, a] = -50
        else:
            rew[s, a] = -1

In [29]:
end = np.zeros(N, dtype=bool)
end[:10] = True
end[999] = True

In [31]:
def value_iteration(kind):
    V = np.zeros(N)
    while True:
        Q = rew + GAMMA * V[nxt]       # value of each safe move
        Q[kind == 1] = GAMBLE_VALUE    # gambles get their average
        new_V = Q.max(axis=1)          # pick the best action
        new_V[end] = 0                 # end squares are worth 0
        if np.abs(new_V - V).max() < 1e-6:
            return new_V
        V = new_V

In [32]:
p_gamble = (kind == 1).sum() / (kind >= 0).sum()
rng = np.random.default_rng(0)
RUNS = 200
total = np.zeros(N)
for _ in range(RUNS):
    k = kind.copy()
    unknown = k == -1
    k[unknown] = rng.random(unknown.sum()) < p_gamble
    total += value_iteration(k)
V = total / RUNS

In [33]:
out = pd.DataFrame({'subtaskID': 1,
                    'datapointID': queries['id'],
                    'answer': V[queries['state_id']]})
out.to_csv('predictions.csv', index=False)